# Workshop: Data Reading & File Formats — Guide

## Scenario

An analytics team needs to ingest raw data from three upstream systems into a Bronze Delta Lake layer.
Data arrives in three formats: customer records as CSV, order events as JSON, and product catalog as Parquet.

Your goal is to read each format using the appropriate Spark reader approach, understand the performance
trade-offs between schema inference and manual schemas, expose files as SQL-native tables via `read_files()`,
and create two Bronze managed Delta tables with CTAS — including lightweight transformations.

## Key Concepts

### DataFrameReader — entry point for all batch reads

`spark.read` returns a `DataFrameReader`. Chain methods to configure the read, then call `.load(path)` to trigger it:

```python
df = (spark.read
      .format("csv")                   # csv | json | parquet | delta | ...
      .option("header", "true")        # format-specific options
      .option("inferSchema", "true")   # or: .schema(mySchema)
      .load("/path/to/file.csv"))      # action — triggers the read
```

Every `.option()` returns the same `DataFrameReader` so calls chain freely.

---

### inferSchema — the two-scan penalty

When `inferSchema=true`, Spark makes **two filesystem passes**:

1. **Scan 1** — reads the entire file to sample values and detect column types
2. **Scan 2** — reads the file again with the inferred schema to build the DataFrame

On cloud object storage (ADLS, S3, GCS) each scan costs I/O and time.
On a 100 GB CSV file this doubles the read cost and execution time.

**Production rule:** always define a manual `StructType` schema in production pipelines.
`inferSchema=true` is acceptable only for one-off exploration or prototyping.

---

### StructType schema definition

```python
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType

schema = StructType([
    StructField("customer_id",       StringType(),    nullable=False),
    StructField("registration_date", TimestampType(), nullable=True),
    StructField("total_amount",      DoubleType(),    nullable=True),
])

df = spark.read.format("csv").option("header", "true").schema(schema).load(path)
```

- One scan — schema is known upfront, no sampling needed
- Raises an error if a `nullable=False` field contains nulls
- Type coercion at read time: a string `"2023-01-15"` becomes `TimestampType` automatically

**Partial schema with JSON:** if your schema lists fewer columns than the JSON record contains,
Spark silently ignores the unlisted fields — useful for selecting only the columns you need.

---

### read_files() — SQL-native Volume reader (DBR 13.3+)

Reads files directly in SQL without a Python DataFrame:

```sql
SELECT *, _metadata.file_path AS file_path
FROM read_files('/Volumes/catalog/schema/volume/data.csv',
                format => 'csv',
                header => true)
```

Note the `=>` arrow syntax for named arguments (not `=`).
`_metadata` struct columns are available automatically: `file_path`, `file_name`, `file_size`.

---

### CTAS — Create Table As Select

Creates a managed Delta table from a SELECT result in one statement:

```sql
CREATE OR REPLACE TABLE catalog.schema.target_table AS
SELECT
    UPPER(first_name)                            AS first_name,
    LOWER(email)                                 AS email,
    TO_DATE(reg_date, 'yyyy-MM-dd')              AS reg_date,
    DATEDIFF(current_date(), reg_date)           AS days_since,
    current_timestamp()                          AS _ingestion_timestamp,
    _metadata.file_path                          AS _source_file
FROM read_files('/path/to/file.csv', format => 'csv', header => true)
WHERE country IS NOT NULL
```

- `CREATE OR REPLACE` is **idempotent** — drops and recreates atomically on each run
- Schema is derived from the SELECT column list — no DDL needed
- **Not incremental** — every run replaces the full table (use Auto Loader / COPY INTO for incremental)

## Prerequisites

- Cluster is running and attached to the notebook
- Run the **Setup** cell first — it sets `CATALOG`, `BRONZE_SCHEMA`, `DATASET_PATH`, and the three file path variables
- Source files: `customers/customers.csv`, `orders/orders_batch.json`, `products/products.parquet` (inside `DATASET_PATH`)
- `from pyspark.sql.types import *` and `import time` are already imported in the configuration cell — do not re-import

## Tasks Overview

| Task | Topic | What you will do |
|------|-------|-----------------|
| 1 | CSV — inferSchema | Read `customers.csv` with `inferSchema=true`, inspect inferred types |
| 2 | CSV — manual schema | Re-read with `StructType`, compare types — one scan instead of two |
| 3 | JSON — partial schema | Define 5-column schema for orders JSON, ignoring remaining fields |
| 4 | Parquet | Read `products.parquet` — no schema needed, types come from the file footer |
| 5 | Performance comparison | Time both approaches with `time.time()` + `.count()`, compare results |
| 6 | read_files() | Read CSV via SQL `read_files()`, expose `_metadata.file_path` |
| 7 | CTAS basic | Create `orders_bronze` Delta table from JSON, add `_source_file` |
| 8 | CTAS + transforms | Create `customers_bronze` with UPPER/LOWER, TO_DATE, DATEDIFF, filter |
| **Bonus 1** | **Bug hunt** | Fix broken JSON read where `inferSchema=false` causes wrong column types |
| **Bonus 2** | **Function design** | Implement idempotent `ingest_to_bronze()` for all three formats |

## Hints — Tasks 1–8

### Task 1 — CSV with inferSchema

```python
customers_auto_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(CUSTOMERS_CSV)
)
customers_auto_df.printSchema()
display(customers_auto_df.limit(5))
```

Check the inferred types — Spark usually gets `StringType` for mixed-value columns and may infer `TimestampType` or `DateType` for date strings depending on the format.

---

### Task 2 — CSV with manual StructType

```python
customers_schema = StructType([
    StructField("customer_id",       StringType(),    nullable=False),
    StructField("first_name",        StringType(),    nullable=True),
    StructField("last_name",         StringType(),    nullable=True),
    StructField("city",              StringType(),    nullable=True),
    StructField("email",             StringType(),    nullable=True),
    StructField("country",           StringType(),    nullable=True),
    StructField("registration_date", TimestampType(), nullable=True),
])

customers_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .schema(customers_schema)
    .load(CUSTOMERS_CSV)
)
```

Compare `customers_df.dtypes` with `customers_auto_df.dtypes` — they should match. If `registration_date` was inferred as `StringType` in Task 1, the manual schema forces `TimestampType`.

---

### Task 3 — JSON with partial schema

```python
orders_schema = StructType([
    StructField("order_id",        StringType(), nullable=True),
    StructField("customer_id",     StringType(), nullable=True),
    StructField("order_datetime",  StringType(), nullable=True),  # keep as String in Bronze
    StructField("total_amount",    DoubleType(), nullable=True),
    StructField("payment_method",  StringType(), nullable=True),
])

orders_df = (
    spark.read
    .format("json")
    .schema(orders_schema)
    .load(ORDERS_JSON)
)
```

With a partial schema, Spark reads **only** the 5 listed fields and discards everything else.
Check `len(orders_df.columns) == 5` — the validation cell confirms this.

---

### Task 4 — Parquet (no schema needed)

```python
products_df = (
    spark.read
    .format("parquet")
    .load(PRODUCTS_PARQUET)
)
products_df.printSchema()
```

The schema is embedded in the Parquet file footer — Spark reads it before loading any rows.
No `.option()` or `.schema()` calls needed.

---

### Task 5 — Performance comparison

Pattern: wrap an **action** (`.count()`) with `time.time()`:

```python
import time

# inferSchema read
start_auto    = time.time()
df_auto       = spark.read.format("csv").option("header","true").option("inferSchema","true").load(CUSTOMERS_CSV)
count_auto    = df_auto.count()          # action — forces execution
time_auto     = time.time() - start_auto

# Manual schema read
start_manual  = time.time()
df_manual     = spark.read.format("csv").option("header","true").schema(customers_schema).load(CUSTOMERS_CSV)
count_manual  = df_manual.count()
time_manual   = time.time() - start_manual
```

On a small training dataset the absolute difference is small due to JVM warmup and local caching.
On 100 GB files in cloud storage the difference is measured in **minutes and dollars**.

---

### Task 6 — read_files() with _metadata

```python
customers_rf_df = spark.sql(f"""
    SELECT *, _metadata.file_path AS file_path
    FROM read_files('{CUSTOMERS_CSV}',
                    format => 'csv',
                    header => true)
""")
display(customers_rf_df.limit(5))
```

Note the `=>` arrow for named arguments (SQL syntax, not Python keyword arguments).
`_metadata` is a hidden struct column — it is not in `*` by default, so alias it explicitly.

---

### Task 7 — CTAS basic (orders_bronze)

```python
spark.sql(f"""
    CREATE OR REPLACE TABLE {ORDERS_BRONZE} AS
    SELECT
        *,
        _metadata.file_path AS _source_file
    FROM read_files('{ORDERS_JSON}', format => 'json')
""")
display(spark.table(ORDERS_BRONZE).limit(5))
```

`CREATE OR REPLACE` makes this idempotent — running it twice produces the same result.
The `_source_file` column records which file each row came from.

---

### Task 8 — CTAS with transforms (customers_bronze)

```python
spark.sql(f"""
    CREATE OR REPLACE TABLE {CUSTOMERS_BRONZE} AS
    SELECT
        customer_id,
        UPPER(first_name)                           AS first_name,
        UPPER(last_name)                            AS last_name,
        city,
        LOWER(email)                                AS email,
        country,
        TO_DATE(registration_date, 'yyyy-MM-dd')    AS registration_date,
        DATEDIFF(current_date(), registration_date) AS days_since_registration,
        current_timestamp()                         AS _ingestion_timestamp
    FROM read_files('{CUSTOMERS_CSV}', format => 'csv', header => true)
    WHERE country IS NOT NULL
""")
```

`DATEDIFF(end, start)` — note the argument order: end date first, start date second.
`current_timestamp()` requires no arguments — parentheses must still be present.

## Hints — Bonus Tasks

### Challenge 1 — Bug Hunt: Wrong Schema

The broken code:

```python
bad_df = (spark.read
    .format("json")
    .option("inferSchema", "false")
    .load(ORDERS_JSON))
# total_amount is StringType instead of DoubleType
```

**Root cause:** For JSON, `inferSchema=false` (the default) means Spark reads **all values as strings**.
Unlike CSV where `inferSchema=false` simply skips sampling, JSON without type detection produces an all-string schema.

**Fix option 1 — pass a manual schema:**
```python
fixed_df = (spark.read
    .format("json")
    .schema(orders_schema)          # orders_schema from Task 3
    .load(ORDERS_JSON))
assert dict(fixed_df.dtypes)["total_amount"] == "double"
```

**Fix option 2 — enable inferSchema:**
```python
fixed_df = (spark.read
    .format("json")
    .option("inferSchema", "true")
    .load(ORDERS_JSON))
assert dict(fixed_df.dtypes)["total_amount"] == "double"
```

**Key insight:** In JSON, `inferSchema=false` and `inferSchema=true` have opposite defaults from CSV:
- CSV default: `inferSchema=false` → all strings
- JSON default: all fields become strings unless you either set `inferSchema=true` or pass a schema

---

### Challenge 2 — Multi-Format Bronze Ingestion Function

Design guidance:

```python
def ingest_to_bronze(dataset_path: str, catalog: str, schema: str) -> dict:
    """
    Read customers.csv, orders_batch.json, and products.parquet from dataset_path.
    Write three Bronze Delta tables with _source_file and _ingestion_timestamp.
    Idempotent — safe to call multiple times (uses CREATE OR REPLACE).
    
    Returns: dict mapping table name -> row count
    """
    results = {}
    
    # 1. Customers CSV — use manual schema + CTAS
    spark.sql(f"""
        CREATE OR REPLACE TABLE {catalog}.{schema}.customers_bronze2 AS
        SELECT *, _metadata.file_path AS _source_file, current_timestamp() AS _ingestion_timestamp
        FROM read_files('{dataset_path}/customers/customers.csv', format => 'csv', header => true)
    """)
    results["customers_bronze2"] = spark.table(f"{catalog}.{schema}.customers_bronze2").count()
    
    # 2. Orders JSON
    spark.sql(f"""
        CREATE OR REPLACE TABLE {catalog}.{schema}.orders_bronze2 AS
        SELECT *, _metadata.file_path AS _source_file, current_timestamp() AS _ingestion_timestamp
        FROM read_files('{dataset_path}/orders/orders_batch.json', format => 'json')
    """)
    results["orders_bronze2"] = spark.table(f"{catalog}.{schema}.orders_bronze2").count()
    
    # 3. Products Parquet
    spark.sql(f"""
        CREATE OR REPLACE TABLE {catalog}.{schema}.products_bronze2 AS
        SELECT *, _metadata.file_path AS _source_file, current_timestamp() AS _ingestion_timestamp
        FROM read_files('{dataset_path}/products/products.parquet', format => 'parquet')
    """)
    results["products_bronze2"] = spark.table(f"{catalog}.{schema}.products_bronze2").count()
    
    return results

# Run it
counts = ingest_to_bronze(DATASET_PATH, CATALOG, BRONZE_SCHEMA)
print(counts)
```

Idempotency comes from `CREATE OR REPLACE` — calling the function twice produces the same three tables.

## Summary

| Task | Format | Key Takeaway |
|------|--------|-------------|
| 1 | CSV | `inferSchema=true` detects types automatically — 2 filesystem scans |
| 2 | CSV | `StructType` schema — 1 scan, guaranteed types, production-ready |
| 3 | JSON | Partial schema reads only listed columns — unused fields silently ignored |
| 4 | Parquet | Self-describing — schema in file footer, no options needed |
| 5 | CSV | Manual schema is faster; gap grows with file size (minutes on 100 GB) |
| 6 | CSV (SQL) | `read_files()` = SQL-native reader; `_metadata.file_path` tracks origin |
| 7 | JSON → Delta | CTAS creates a managed Delta table; `CREATE OR REPLACE` is idempotent |
| 8 | CSV → Delta | Type casts, computed columns, and filters all inside a CTAS SELECT |

**Format decision guide:**

| Format | Schema needed? | Use case |
|--------|---------------|----------|
| CSV | Manual (production), `inferSchema` (exploration) | Flat tabular exports, spreadsheets |
| JSON | Manual (partial or full), or `inferSchema=true` | API payloads, nested/semi-structured |
| Parquet | None — embedded in footer | Columnar analytics, Lakehouse storage |

> **Next step:** For *incremental* ingestion (new files arriving over time) — see **Lab 05: Streaming & Auto Loader** (Day 2).

← [02 — Data Reading & File Formats](../day1/demo/02_data_ingestion.ipynb) | **[README](../../README.md)** | [03 — Delta Lake →](../day1/demo/03_delta_lake.ipynb)